# PDE simulation of bullet options

This project deals with the numerical resolution of a parabolic PDE with a Monte-Carlo method, by harvesting massive parallel programming on GPU devices.

The Feynman–Kac formula establishes a clear, modern link between parabolic partial differential equations (PDEs) and stochastic differential equations (SDEs) by expressing the solutions of the latter as expectations taken over paths that follow the former.

We fix a time interval $[0, T]$, and note $t, s$ two time increments such that $0\leq s \leq t\leq T$.

In this work, in accordance with Black-Scholes model, it will be assumed that the asset price $(S_t)_{t\in [0, T]}$ follows the time evolution law:
\begin{equation}
S_t=S_s\exp((r-\sigma^2/2)(t-s) + \sigma \sqrt(t-s) G),
\end{equation}
where $r$ is the risk-free rate, $\sigma$ is the SDE's volatility and $G$ is a random variable following the standard normal law $N(0, 1)$.

## Part 1

# PDE simulation of bullet options

This project deals with the numerical resolution of a parabolic PDE with a Monte-Carlo method, by harvesting massive parallel programming on GPU devices.

The Feynman–Kac formula establishes a clear, modern link between parabolic partial differential equations (PDEs) and stochastic differential equations (SDEs) by expressing the solutions of the latter as expectations taken over paths that follow the former.

We fix a time interval $[0, T]$, and note $t, s$ two time increments such that $0\leq s \leq t\leq T$.

In this work, in accordance with Black-Scholes model, it will be assumed that the asset price $(S_t)_{t\in [0, T]}$ follows the time evolution law:
\begin{equation}
S_t=S_s\exp((r-\sigma^2/2)(t-s) + \sigma \sqrt(t-s) G),
\end{equation}
where $r$ is the risk-free rate, $\sigma$ is the SDE's volatility and $G$ is a random variable following the centered standard normal law $N(0, 1)$.

Bullet option requires discretizing the time interval into $M+1$ sub-interval with an uniformly spaced time schedule $T_0=0 < T_1 < T_2 < \cdots < T_{M} < T_{M+1}=T$, so that $\forall i \in \{1, \cdots, M+1\}, T_i=\frac{i}{M+1}T=i\Delta t$, $\Delta t=\frac{i}{M+1}$.

The first objective is to propose a reliable and efficient code to estimate the price of a bullet option: $F(t, x, j)=e^{-r(T-t)}\mathbb{E}_{(S_u)_{u\in [t, T]}}[X|S_t=x, I_t=j]$, with $X=(S_T - K)_+ 1_{I_T\in [P_1, P_2]}$, and define for all $t \in [0, T], I_t=\sum\limits_{T_i \leq t} 1_{S_{T_i} < B}$. Note that by construction, $I_t\leq \#\{T_i \leq t\}$, i.e. $j$ must take value in $\{0, 1, \cdots, i\}$. The variable $B$ is commonly called the barrier.

As $I_T$ depends on the path history at instants $(T_i)_i$, simulating directly $S_T$ from $S_{T_i}$ only is insufficient for computing $X$; the asset price at each intermediate time must be simulated. That's what the `for` loop do in the algorithm below.

We now have everything we need to write the algorithm that computes $F(T_i, x, j)$, where $i \in \{0, 1, \dots, M-1, M\}$.

> Data: $i (\text{or }T_i), x, j, B, P_1, P_2$
> for $m=1:\text{Nb}_{\text{traj}}$
>> for $k=i:N$
>>> sample $G \sim N(0, 1)$
>>> compute $S_{T_{k+1}}=S_{T_j}\exp((r-\sigma^2/2)(T_{k+1}-T_{k}) + \sigma \sqrt(T_{k+1} - T_k) G)$
>>> $I_{T_{k+1}}=I_{T_k} + 1_{S_{T_{k+1}} \leq B}$

>> Calculate $X_m=(S_T-K)_+ 1_{I_{T} \in [P_1, P_2]}$

> Compute the mean $F(T_i, x, j)=\frac{e^{-r(T-T_i)}}{\text{Nb}_{\text{traj}}}\sum\limits_{m=1}^{\text{Nb}_{\text{traj}}} X_m$

There are many possible approaches to estimating $F$ in CUDA, depending on how the loops and parameters are parallelised or serialised in the algorithm. Each choice may affect execution time on the device, for better or worse.

The `for` loop over $m$ is always included in the kernel; we must parallelize over trajectories to fully leverage the GPU capabilities.

Describe here the implementations proposed. I see $3$ of them: the trash implementation, the MC_k4 implementation and the one calling a `__device__` function.

A loop iterating over $T_i$ should be implemented in the host.

Future Works: write a CUDA extension for Python.

Alternatively, one could also parallelize over $T_i$ inside the kernel, in a nested fashion by harvesting a second axis of the grid.

The two versions are implemented, respectively in the kernels `MC` and `MC_nested`.

**Exercice 1** Assuming $B = 110$, $M = 100$, $P_1 = 10$, $P_2 = 40$ and $Ti = i/M$ for $i \in \{0, ..., M\}$ , write down a Monte Carlo simulation code to approximate $F (Tk , x, j)$. (8 points)

In [ ]:
!make

In [ ]:
!./project_exo1

## Part 2

Let $u(t, x, j) = e^{r(T-t)} F(t, e^x, j)$ where $F$ is the price of a bullet option as before. One can then show that, on any interval $t \in [T_{M-k}, T_{M-k+1})$, $k = M, \ldots, 0$, $u(t, x, j)$ is the solution of the PDE

$$
\frac{\partial u}{\partial t}(t, x, j)
+ \mu \frac{\partial u}{\partial x}(t, x, j)
+ \frac{1}{2}\sigma^2 \frac{\partial^2 u}{\partial x^2}(t, x, j)
= 0
$$

with

$$
\mu = r - \frac{\sigma^2}{2}.
$$

The final and boundary conditions are:

- $u(T, x, j) = \max(e^x - K, 0)\mathbf{1}_{\{j \in [P_1, P_2]\}}$ for any $(x, j)$
- $u(t, \log[2K/5], j) = p_{\min} = 0$
- $u(t, \log[5K/2], j) = p_{\max} = \frac{3K}{2}\mathbf{1}_{\{j \in [P_1, P_2]\}}$

From now on we use notations $u_t(x, j) = u(t, x, j)$ and $u_{k,i} = u(t_k, x_i, j)$. Following the Crank--Nicolson scheme, we get

$$
q_u u_{k,i+1} + q_m u_{k,i} + q_d u_{k,i-1}
=
p_u u_{k+1,i+1} + p_m u_{k+1,i} + p_d u_{k+1,i-1}
$$

with

$$
q_u = -\frac{\sigma^2 \Delta t}{4\Delta x^2} - \frac{\mu \Delta t}{4\Delta x},
\qquad
q_d = -\frac{\sigma^2 \Delta t}{4\Delta x^2} + \frac{\mu \Delta t}{4\Delta x},
\qquad
q_m = 1 + \frac{\sigma^2 \Delta t}{2\Delta x^2},
$$

$$
p_u = \frac{\sigma^2 \Delta t}{4\Delta x^2} + \frac{\mu \Delta t}{4\Delta x},
\qquad
p_d = \frac{\sigma^2 \Delta t}{4\Delta x^2} - \frac{\mu \Delta t}{4\Delta x},
\qquad
p_m = 1 - \frac{\sigma^2 \Delta t}{2\Delta x^2}.
$$

The figure below shows an example of how PDE’s backward resolution algorithm (with $M = 10$, $P_1 = 3$, $P_2 = 8$) is deployed with time on the x-axis and the set of values of $I_t$ in the ordinate.

**Exercise 2**Write the kernel that solves the PDE on $[T_{M-1} , T )$ and compare the numerical solution to the Monte
Carlo values of $F (T_{M-1} , x, j)$. (4 points + 2 points)
Use $\sigma=0.2$, $r=0.1$, $K=100$, $T=1$, $M=100$, $P1=10$, $P2=40$, $T_{M-1}=T\frac{M-1}{M}$

In [ ]:
!nvcc project_exo2.cu -arch=compute_60 -code=sm_75 -o project_exo2

In [ ]:
!./project_exo2

In [ ]:
!nvcc project_exo1.cu -arch=compute_60 -code=sm_75 -o project_exo1

In [ ]:
!./project_exo1 1

## Part 3

In the end, because of the discontinuity of the process $I$ at times $T_{M-k}$, we need to set

$$
u_{T_{M-k}}(S_{T_{M-k}}, j) =
\begin{cases}
u_{T_{M-k}}(S_{T_{M-k}}, P_2)\mathbf{1}_{\{S_{T_{M-k}} \ge B\}} & \text{if } j = P_2, \\
u_{T_{M-k}}(S_{T_{M-k}}, P_{k1})\mathbf{1}_{\{S_{T_{M-k}} < B\}} & \text{if } j = P_{k1} - 1, \\
\begin{pmatrix}
u_{T_{M-k}}(S_{T_{M-k}}, j)\mathbf{1}_{\{S_{T_{M-k}} \ge B\}} \\
+ u_{T_{M-k}}(S_{T_{M-k}}, j+1)\mathbf{1}_{\{S_{T_{M-k}} < B\}}
\end{pmatrix} & \text{if } j \in [P_{k1}, P_2 - 1].
\end{cases}
\tag{6}
$$

with

$$
P_{k1} = \max(P_1 - k, 0).
$$

In [1]:
file=open("option_price.txt", "r")

file.close()

FileNotFoundError: [Errno 2] No such file or directory: 'option_price.txt'